# Previsão de Household Income por município (até 2026)

**Objetivo**: a partir do ficheiro `household-income-15-21.parquet`, projetar valores de *household income* para 2022–2026, município a município.

O crescimento anual de cada município é calculado com base em três componentes com pesos fixos:
- **50%** – CAGR histórico do próprio município (crescimento pré-COVID, até 2019)
- **20%** – Fatores macroeconómicos: inflação e crescimento do PIB (10% cada)
- **30%** – Crescimento salarial: salário mínimo e salário médio (15% cada)

**Nota**: este notebook foi gerado em 2026-05-16.

## Fontes (para defender o trabalho)

Não é feito forecast da inflação/salário mínimo; são usadas séries publicadas.

- Salário mínimo Mensal nacional (tabela anual, inclui 2026): https://www.pordata.pt/portugal/evolucao+do+salario+minimo+nacional-74

- Salário Médio Líquido Mensal Em portugal - https://pt.tradingeconomics.com/portugal/wages

- Previsão macro (PIB) – PORDATA - https://www.pordata.pt/portugal/taxa+de+crescimento+do+pib-2298
    - Estimativa para 2026 https://www.bportugal.pt/publicacao/boletim-economico-marco-2026

-Inflação segundo Banco de Portugal - https://bpstat.bportugal.pt/serie/12704650
    - Estimativa para 2026: https://pt.tradingeconomics.com/portugal/inflation-cpi


- (Proxy para mediana) ‘Rendimento mediano disponível’ na PORDATA Retratos da Europa: https://retratos.europa.pordata.pt/custo-de-vida/portugal


In [13]:
# Imports
import pandas as pd
import numpy as np


In [14]:
df = pd.read_parquet('household-income-15-21.parquet')

df

,ANO,Município,Household Income (EUR)
0,2015,Abrantes,15108
1,2015,Aguiar da Beira,11334
2,2015,Alandroal,11515
3,2015,Albergaria-a-Velha,14776
4,2015,Albufeira,13363
...,...,...,...
2354,2021,Área Metropolitana de Lisboa,23321
2355,2021,Área Metropolitana do Porto,20110
2356,2021,Évora,22087
2357,2021,Ílhavo,20196


## 0) Análise de municípios: Ler o Dataset e normalizar colunas

De parquet para dataframe e normalização de colunas para evitar erros no pipeline.

In [15]:
# De parquet para dataframe.
df = pd.read_parquet('household-income-15-21.parquet')



# Mapeamento de nomes alternativos para year/municipio/income.
rename_map = {
    'ANO': 'year',
    'Município': 'municipio',
    'Household Income (EUR)': 'income',
}

# Intera-se entre pares do rename_map para renomear colunas correspondentes.
for k, v in rename_map.items():
    if k in df.columns:
        df = df.rename(columns={k: v})

# Conversão de tipos.
df['year'] = df['year'].astype(int)
df['municipio'] = df['municipio'].astype(str)
df['income'] = pd.to_numeric(df['income'], errors='coerce')

# Ordenação por município e ano para cálculo temporal subsequente.
df = df.sort_values(['municipio', 'year']).reset_index(drop=True)

df.head(8)

,year,municipio,income
0,2015,Abrantes,15108
1,2016,Abrantes,15609
2,2017,Abrantes,16014
3,2018,Abrantes,16574
4,2019,Abrantes,17115
5,2020,Abrantes,17326
6,2021,Abrantes,18005
7,2015,Aguiar da Beira,11334


## 1) Calcular CAGR pré-COVID (até 2019)

CAGR significa Compound Annual Growth Rate. O objetivo é calcular o crescimento anual composto de cada município até 2019, evitando o efeito do choque COVID.

A lógica é fixar o município e medir a variação entre os pontos inicial e final do período pré-COVID. Por exemplo, se o income passa de 100 para 121 em 2 anos, o CAGR é (121/100)^(1/2) - 1 = 0,10, ou 10% ao ano.

In [16]:
def calculate_pre_covid_cagr(group: pd.DataFrame) -> float:
    """Calculate CAGR for one municipio using observations up to 2019."""
    """
    Args:
        group: DataFrame slice for a single municipio (all years for that municipio).
               The caller uses `df.groupby('municipio')`, so each `group` contains only one municipio,
               e.g. rows for 'Abrantes' from 2015..2019. This function will further filter to years <= 2019.

    Returns:
        CAGR as float (e.g. 0.10 for 10%/year) or np.nan if not computable.
    """
    # Filtra até 2019 para estimar crescimento estrutural sem o efeito COVID.
    g = group[group['year'] <= 2019].copy()

    # São necessários pelo menos dois pontos no tempo para calcular CAGR.
    # Exemplo: para Abrantes pode haver linhas para 2015,2016,2017,2018,2019 -> shape[0] >= 2 ok.
    if g.shape[0] < 2:
        return np.nan

    # Pegamos o primeiro e o último valor observados no período filtrado (mesmo municipio).
    start = g.iloc[0]['income']
    end = g.iloc[-1]['income']
    # Número de anos entre os dois pontos (p.ex. 2019 - 2015 = 4).
    years = g.iloc[-1]['year'] - g.iloc[0]['year']



    # Fórmula do CAGR: (end/start)^(1/years) - 1.
    # Exemplo: start=100, end=121, years=2 => (121/100)^(1/2) - 1 = 0.10 (10% ao ano).
    return (end / start) ** (1 / years) - 1

cagr_df = (
    df.groupby('municipio', as_index=False)
      .apply(lambda g: pd.Series({'cagr_pre_covid': calculate_pre_covid_cagr(g)}))
)

cagr_df.head()

,municipio,cagr_pre_covid
0,Abrantes,0.031674
1,Aguiar da Beira,0.056820
2,Alandroal,0.048184
3,Albergaria-a-Velha,0.034377
4,Albufeira,0.034842


## 2) Tabela Macro com fatores económicos para extrapolar os Median Household Income (inflação, PIB, salários)

Dados com bases citadas acima.

- O ambiente do notebook pode não ter acesso à Internet, por isso as séries são definidas manualmente e validadas.
- É verificado que todos os anos necessários estão presentes para evitar buracos no modelo.

**Como preencher**: copiar os valores dos sites e substituir nos dicionários abaixo.

In [17]:
YEARS_FORECAST = [2022, 2023, 2024, 2025, 2026]

# Inflação anual (decimal). Fonte: INE / Banco de Portugal.
inflation = {2022: 0.0783, 2023: 0.0431, 2024: 0.0242, 2025: 0.0234, 2026: 0.0330}

# Crescimento real do PIB (decimal). Fonte: Banco de Portugal / OCDE.
gdp_growth = {2022: 0.070, 2023: 0.031, 2024: 0.022, 2025: 0.019, 2026: 0.018}

# Salário mínimo nacional (EUR/mês). Fonte: PORDATA.
min_wage = {2022: 705, 2023: 760, 2024: 820, 2025: 870, 2026: 920}

# Salário médio líquido (EUR/mês). Fonte: Trading Economics / PORDATA.
avg_wage = {2022: 1010, 2023: 1090, 2024: 1230, 2025: 1310, 2026: 1370}

# Construir tabela macro
macro = pd.DataFrame({
    'year': YEARS_FORECAST,
    'inflation': [inflation[y] for y in YEARS_FORECAST],
    'gdp_growth': [gdp_growth[y] for y in YEARS_FORECAST],
    'min_wage':   [min_wage[y]   for y in YEARS_FORECAST],
    'avg_wage':   [avg_wage[y]   for y in YEARS_FORECAST],
})

# Calcular taxas de crescimento anuais dos salários
macro['g_min_wage'] = macro['min_wage'].pct_change().fillna(0)
macro['g_avg_wage'] = macro['avg_wage'].pct_change().fillna(0)

macro

,year,inflation,gdp_growth,min_wage,avg_wage,g_min_wage,g_avg_wage
0,2022,0.0783,0.070,705,1010,0.000000,0.000000
1,2023,0.0431,0.031,760,1090,0.078014,0.079208
2,2024,0.0242,0.022,820,1230,0.078947,0.128440
3,2025,0.0234,0.019,870,1310,0.060976,0.065041
4,2026,0.0330,0.018,920,1370,0.057471,0.045802


## 3) Preparar base (último ano observado) e juntar CAGR

O ponto de partida para a projeção é o income de cada município em 2021 (último ano observado), ao qual se junta o CAGR calculado na secção anterior.

## 4) Loop de projeção (2022–2026)

Fórmula de crescimento anual com pesos fixos:

```
g_total = 0.50 * cagr_pre_covid
        + 0.10 * inflação
        + 0.10 * crescimento_pib
        + 0.15 * crescimento_salario_minimo
        + 0.15 * crescimento_salario_medio
```

A projeção aplica este `g_total` ao income do ano anterior, iterando de 2022 a 2026.

In [18]:
latest_year = int(df['year'].max())

base_df = df[df['year'] == latest_year].copy()
base_df = base_df.merge(cagr_df, on='municipio', how='left')

base_df.head()

,year,municipio,income,cagr_pre_covid
0,2021,Abrantes,18005,0.031674
1,2021,Aguiar da Beira,15269,0.056820
2,2021,Alandroal,14850,0.048184
3,2021,Albergaria-a-Velha,18064,0.034377
4,2021,Albufeira,15233,0.034842


In [19]:
projections = []
current_df = base_df.copy()

for _, row in macro.sort_values('year').iterrows():
    year = int(row['year'])

    next_df = current_df.copy()

    # Taxa de crescimento anual = média ponderada dos cinco fatores
    g_total = (
        0.50 * next_df['cagr_pre_covid'] +  # 50% tendência local pré-COVID
        0.10 * float(row['inflation'])    +  # 10% inflação
        0.10 * float(row['gdp_growth'])   +  # 10% crescimento do PIB
        0.15 * float(row['g_min_wage'])   +  # 15% crescimento do salário mínimo
        0.15 * float(row['g_avg_wage'])      # 15% crescimento do salário médio
    )

    next_df['income'] = next_df['income'] * (1 + g_total)
    next_df['year'] = year

    projections.append(next_df[['municipio', 'year', 'income']])
    current_df = next_df

projection_df = pd.concat(projections, ignore_index=True)

final_df = pd.concat(
    [df[['municipio', 'year', 'income']], projection_df],
    ignore_index=True
).sort_values(['municipio', 'year']).reset_index(drop=True)



final_df.head(20)

,municipio,year,income
0,Abrantes,2015,15108.000000
1,Abrantes,2016,15609.000000
2,Abrantes,2017,16014.000000
3,Abrantes,2018,16574.000000
4,Abrantes,2019,17115.000000
5,Abrantes,2020,17326.000000
6,Abrantes,2021,18005.000000
7,Abrantes,2022,18557.159252
8,Abrantes,2023,19426.196795
9,Abrantes,2024,20427.911658


## 5) Sanity checks rápidos

Top 10 municípios em 2026 e série temporal de Lisboa para validar a projeção.

In [20]:
# Top 10 municípios em 2026
final_df[final_df['year'] == 2026].sort_values('income', ascending=False).head(10)


,municipio,year,income
1655,Lisboa,2026,36168.432067
2279,Oeiras,2026,35695.231906
95,Alcochete,2026,32638.780358
899,Cascais,2026,31989.700650
2699,Porto,2026,31735.866756
1067,Coimbra,2026,30794.915006
1763,Mafra,2026,28487.374129
515,Arruda dos Vinhos,2026,28408.285461
3995,Área Metropolitana de Lisboa,2026,28107.810446
983,Castro Verde,2026,28034.719538


In [21]:
final_df[final_df['municipio'] == 'Lisboa'].sort_values('year')



,municipio,year,income
1644,Lisboa,2015,24337.000000
1645,Lisboa,2016,24794.000000
1646,Lisboa,2017,25169.000000
1647,Lisboa,2018,27777.000000
1648,Lisboa,2019,28372.000000
1649,Lisboa,2020,28072.000000
1650,Lisboa,2021,29082.000000
1651,Lisboa,2022,30081.782884
1652,Lisboa,2023,31602.157724
1653,Lisboa,2024,33349.005712


In [22]:
final_df.set_index("year")

,municipio,income
year,,
2015,Abrantes,15108.000000
2016,Abrantes,15609.000000
2017,Abrantes,16014.000000
2018,Abrantes,16574.000000
2019,Abrantes,17115.000000
...,...,...
2022,Óbidos,18648.723943
2023,Óbidos,19577.276267
2024,Óbidos,20644.758182


In [23]:
final_df.to_parquet('household_income_projection_2026.parquet', index=True)